# Datas e fusos horários

Use datas para prazos, duração e agrupamento por período. UTC é uma referência; converter para uma cidade muda a representação do mesmo instante. Use America/Sao_Paulo e America/Cuiaba, que consideram regras por data, em vez de subtrair horas manualmente. Em exemplos de outubro de 2026, 15h UTC corresponde a 12h em São Paulo e 11h em Cuiabá.

Execute as demonstrações; depois tente os exercícios antes do gabarito. Escreva dados disponíveis, resultado esperado e passos em português. Requisitos: Python 3.9 ou superior, pandas e IPython. Para fusos, o ambiente precisa de dados IANA; se houver ZoneInfoNotFoundError, instale tzdata no kernel. Não é necessário baixar datasets.

In [ ]:
import pandas as pd
from IPython.display import display

## Demo 1 Interpretar datas
format informa o padrão; errors="coerce" transforma dados inválidos em NaT. A divisão por dias mede duração, não mudança de data no calendário.

In [2]:
pedidos = pd.DataFrame({"pedido": [1,2,3], "inicio": ["01/10/2026","03/10/2026","31/02/2026"], "fim": ["04/10/2026","08/10/2026","09/10/2026"]})
for coluna in ["inicio", "fim"]:
    pedidos[coluna] = pd.to_datetime(pedidos[coluna], format="%d/%m/%Y", errors="coerce")
pedidos["dias"] = (pedidos["fim"]-pedidos["inicio"]).dt.total_seconds()/86400
display(pedidos)

,pedido,inicio,fim,dias
0,1,2026-10-01,2026-10-04,3.0
1,2,2026-10-03,2026-10-08,5.0
2,3,NaT,2026-10-09,NaN


## Demo 2 UTC para duas cidades
Z indica UTC. utc=True interpreta textos sem fuso como UTC também: só faça isso se a origem realmente usar UTC. tz_convert preserva o instante. Primeiro converta, depois extraia a data local.

In [3]:
eventos = pd.DataFrame({"id": [1,2,3,4], "texto_utc": ["2026-10-05T15:00:00Z","2026-10-05T02:30:00Z","2026-10-05T03:30:00Z","invalido"]})
eventos["utc"] = pd.to_datetime(eventos["texto_utc"], format="ISO8601", utc=True, errors="coerce")
eventos["sp"] = eventos["utc"].dt.tz_convert("America/Sao_Paulo")
eventos["cuiaba"] = eventos["utc"].dt.tz_convert("America/Cuiaba")
display(eventos)
assert eventos.loc[0,"sp"].hour == 12
assert eventos.loc[0,"cuiaba"].hour == 11

,id,texto_utc,utc,sp,cuiaba
0,1,2026-10-05T15:00:00Z,2026-10-05 15:00:00+00:00,2026-10-05 12:00:00-03:00,2026-10-05 11:00:00-04:00
1,2,2026-10-05T02:30:00Z,2026-10-05 02:30:00+00:00,2026-10-04 23:30:00-03:00,2026-10-04 22:30:00-04:00
2,3,2026-10-05T03:30:00Z,2026-10-05 03:30:00+00:00,2026-10-05 00:30:00-03:00,2026-10-04 23:30:00-04:00
3,4,invalido,NaT,NaT,NaT


## Demo 3 Localizar não é converter
Uma hora sem fuso não informa um instante único. tz_localize atribui o fuso de origem sem mudar o relógio; tz_convert muda a representação para outro fuso. Aqui a origem declarou que o texto é horário de Cuiabá. Datas históricas de transição podem ser ambíguas ou inexistentes; usamos raise para não escolher silenciosamente.

In [4]:
local = pd.to_datetime(pd.Series(["2026-10-05 09:00:00"]), format="%Y-%m-%d %H:%M:%S")
local_cuiaba = local.dt.tz_localize("America/Cuiaba", ambiguous="raise", nonexistent="raise")
display(pd.DataFrame({"cuiaba":local_cuiaba, "utc":local_cuiaba.dt.tz_convert("UTC"), "sp":local_cuiaba.dt.tz_convert("America/Sao_Paulo")}))

,cuiaba,utc,sp
0,2026-10-05 09:00:00-04:00,2026-10-05 13:00:00+00:00,2026-10-05 10:00:00-03:00


## Exercícios

### 1 Prazos
Liste datas inválidas e pedidos que demoraram mais de 4 dias. Não confunda NaT com prazo zero.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [5]:
# Sua solução aqui.

### 2 Mudança de dia
Conte eventos por data local em SP e Cuiabá. Separe registros inválidos. Explique o evento de 03h30 UTC: em SP é dia 5, em Cuiabá ainda é dia 4.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [6]:
# Sua solução aqui.

### 3 Janela comercial
Selecione eventos entre 08h inclusive e 18h exclusive em cada cidade. Mostre IDs.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [7]:
# Sua solução aqui.

### 4 Reunião e duração
Uma reunião começa às 09h de Cuiabá em 05/10/2026 e dura 90 minutos. Mostre início e fim em UTC e SP e confira a duração.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [8]:
# Sua solução aqui.

## Gabarito comentado

### 1 Prazos

Como pensar: Verificar nulos → selecionar durações válidas acima de 4.

In [9]:
display(pedidos.loc[pedidos["inicio"].isna() | pedidos["fim"].isna()])
display(pedidos.loc[pedidos["dias"] > 4])
assert pedidos.loc[1,"dias"] == 5

,pedido,inicio,fim,dias
2,3,NaT,2026-10-09,NaN


,pedido,inicio,fim,dias
1,2,2026-10-03,2026-10-08,5.0


### 2 Mudança de dia

Como pensar: Separar válidos → converter → extrair data → contar.

In [10]:
validos = eventos.loc[eventos["utc"].notna()].copy()
for cidade in ["sp","cuiaba"]:
    display(validos[cidade].dt.date.value_counts().sort_index())
display(eventos.loc[eventos["utc"].isna()])
assert eventos.loc[2,"sp"].day == 5
assert eventos.loc[2,"cuiaba"].day == 4

sp
2026-10-04    1
2026-10-05    2
Name: count, dtype: int64

cuiaba
2026-10-04    2
2026-10-05    1
Name: count, dtype: int64

,id,texto_utc,utc,sp,cuiaba
3,4,invalido,NaT,NaT,NaT


### 3 Janela comercial

Como pensar: Usar hora local → aplicar duas condições → selecionar.

In [11]:
for cidade in ["sp","cuiaba"]:
    horas = eventos[cidade].dt.hour
    display(eventos.loc[(horas >= 8) & (horas < 18), ["id",cidade]])

,id,sp
0,1,2026-10-05 12:00:00-03:00


,id,cuiaba
0,1,2026-10-05 11:00:00-04:00


### 4 Reunião e duração

Como pensar: Localizar a hora de origem → somar duração → converter.

In [12]:
inicio = pd.Timestamp("2026-10-05 09:00:00").tz_localize("America/Cuiaba")
fim = inicio + pd.Timedelta(minutes=90)
for fuso in ["UTC","America/Sao_Paulo","America/Cuiaba"]:
    print(fuso, inicio.tz_convert(fuso), fim.tz_convert(fuso))
assert (fim-inicio).total_seconds()/60 == 90

UTC 2026-10-05 13:00:00+00:00 2026-10-05 14:30:00+00:00
America/Sao_Paulo 2026-10-05 10:00:00-03:00 2026-10-05 11:30:00-03:00
America/Cuiaba 2026-10-05 09:00:00-04:00 2026-10-05 10:30:00-04:00


# Oficina de funções e reutilização

Definir prepara uma função; chamar executa seu corpo. Em notebooks, execute primeiro a célula com def. Faça os exercícios antes de consultar as soluções. As oficinas usam dados próprios e podem ser executadas após os imports.

## Demo Conversão repetida
Origem declarada UTC. Uma função recebe a Series e o fuso de destino; utc=True não deve ser usado para fingir que uma hora local é UTC.

In [13]:
horarios = pd.Series(["2026-10-05T03:30:00Z","2026-10-05T15:00:00Z","invalido"])
sp_repetido = pd.to_datetime(horarios,format="ISO8601",utc=True,errors="coerce").dt.tz_convert("America/Sao_Paulo")
cuiaba_repetido = pd.to_datetime(horarios,format="ISO8601",utc=True,errors="coerce").dt.tz_convert("America/Cuiaba")

## Pratique a transformação
Para cada atividade, escreva entrada → processamento → saída. Compare o código antes e depois: onde a regra fica definida? Quantos lugares precisam mudar quando ela muda? Funções não tornam todo código mais rápido; o ganho aqui é organização, reutilização e manutenção.

### 1 Extraia a função

Crie converter_utc(textos, destino). Chame para SP e Cuiabá e compare com as duas versões repetidas.

In [14]:

# Sua solução aqui.

### 2 Reutilize o filtro

Crie em_expediente(horarios_locais, inicio=8, fim=18) retornando máscara booleana. Aplique em SP e Cuiabá.

In [15]:

# Sua solução aqui.

## Soluções da oficina

### 1 Extraia a função

A interpretação do texto fica em um lugar; o fuso vira parâmetro. Não escondemos NaT.

In [16]:
def converter_utc(textos,destino):
    instantes = pd.to_datetime(textos,format="ISO8601",utc=True,errors="coerce")
    return instantes.dt.tz_convert(destino)

sp = converter_utc(horarios,"America/Sao_Paulo")
cuiaba = converter_utc(horarios,destino="America/Cuiaba")
assert sp.equals(sp_repetido)
assert cuiaba.equals(cuiaba_repetido)
display(pd.DataFrame({"sp":sp,"cuiaba":cuiaba}))

,sp,cuiaba
0,2026-10-05 00:30:00-03:00,2026-10-04 23:30:00-04:00
1,2026-10-05 12:00:00-03:00,2026-10-05 11:00:00-04:00
2,NaT,NaT


### 2 Reutilize o filtro

A função recebe horas já convertidas. O início é inclusivo e o fim exclusivo; ausentes não entram.

In [17]:
def em_expediente(horarios_locais,inicio=8,fim=18):
    horas = horarios_locais.dt.hour
    return (horas >= inicio) & (horas < fim)

print(em_expediente(sp).tolist())
print(em_expediente(cuiaba).tolist())
assert em_expediente(sp).tolist() == [False,True,False]

[False, True, False]
[False, True, False]


Referência: https://docs.python.org/3/tutorial/controlflow.html#defining-functions . lambda contém uma expressão; use def quando precisar de várias etapas, validação ou uma regra reutilizada com nome.